# Prepare HC dataset for VLM training

Convert the HC clinical-photography dump into canonical JSONL.

Kept as an **experimental / exploration** resource. It is **not** the primary dataset of the main study (PAD-UFES-20).

- **Source:** `data/datasets/HC/` — `lesions.csv` + `images/*.png`
- **Output:** `data/processed/hc/clinical_context/`
- **Image:** cropped file (`imageCropped`); full photos stay unused
- **Splits:** none on disk — 80/10/10 grouped by `lesionId`, stratified by mapped `code`
- **Labels:** `examResult` when present, else `clinicDiagnosis`; mapped to ISIC-like codes when possible
- **Metadata:** phototype, body part, clinical description (source language preserved)
- **No** `benign_malignant` column


## 1. Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Could not find repo root (src/) from {Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

from vlm_ft.data.canonical import hc_image_rel, hc_prompt_and_label
from vlm_ft.data.prepare import (
    count_existing_images,
    load_hc,
    preview_processed,
    print_source_eda,
    rel_to_root,
    require_source,
    rows_to_samples,
    split_by_group,
    write_processed_dataset,
)

CURRENT_SOURCES = "PAD, ISIC18, HC, Derm1M, MILK10K"

HC_ROOT = ROOT / "data/datasets/HC"
OUT_DIR = ROOT / "data/processed/hc/clinical_context"
require_source(HC_ROOT, expected=CURRENT_SOURCES)


## 2. Load


In [ ]:
df = load_hc(HC_ROOT)
print(df.shape)
print("examResult filled:", df["examResult"].notna().mean())
df.head()


## 3. EDA


In [ ]:
ok, total = count_existing_images(df, "image_rel", HC_ROOT)
print(f"cropped images on disk: {ok}/{total}")
print("clinicDiagnosis top:\n", df["clinicDiagnosis"].value_counts(dropna=False).head(15).to_string())

splits = split_by_group(df, group_col="lesionId", label_col="code")
eda = pd.concat(splits.values(), ignore_index=True)
print_source_eda(
    eda,
    label_col="code",
    metadata_cols=["phototype", "bodyPart", "description", "examResult", "clinicDiagnosis"],
)


## 4. Convert to canonical JSONL


In [ ]:
HC_ROOT_REL = rel_to_root(HC_ROOT, ROOT)
samples = {
    split: rows_to_samples(
        frame,
        HC_ROOT,
        prompt_and_label=hc_prompt_and_label,
        image_rel=hc_image_rel,
    )
    for split, frame in splits.items()
}
write_processed_dataset(
    name="hc/clinical_context",
    out_dir=OUT_DIR,
    split_samples=samples,
    image_root_rel=HC_ROOT_REL,
)


## 5. Validate and preview


In [ ]:
preview_processed(OUT_DIR, "hc/clinical_context")
